In [1]:
import numpy as np

class RunningMeanStd:
    """
    Calculates a running mean and variance using Welford's algorithm.
    https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance
    This is numerically stable and avoids recomputing sums over the entire history.
    """
    def __init__(self, epsilon=1e-4, shape=()):
        self.mean = np.zeros(shape, 'float64')
        self.var = np.ones(shape, 'float64')
        self.count = epsilon

    def update(self, x):
        batch_mean = np.mean(x, axis=0)
        batch_var = np.var(x, axis=0)
        batch_count = x.shape[0]

        delta = batch_mean - self.mean
        tot_count = self.count + batch_count

        new_mean = self.mean + delta * batch_count / tot_count
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        m_2 = m_a + m_b + np.square(delta) * self.count * batch_count / tot_count
        new_var = m_2 / tot_count

        self.mean = new_mean
        self.var = new_var
        self.count = tot_count

In [2]:
from Simulation.suite_simple_trading.model import BaseBatteryEnv
import gymnasium as gym
from collections import deque

class NormalizeRewardWrapper(gym.Wrapper):
    """
    A wrapper to normalize rewards based on the running statistics of the returns.

    This wrapper collects rewards during an episode, calculates the discounted returns
    at the end, updates a running mean/variance of these returns, and then scales
    the individual rewards by the running standard deviation of the returns.
    """
    def __init__(self, env: BaseBatteryEnv, gamma: float = 0.99, epsilon: float = 1e-8):
        super().__init__(env)
        # We need a deque to efficiently calculate discounted returns
        self.episode_rewards = deque(maxlen=10000) # Max episode length
        self.return_rms = RunningMeanStd(shape=())
        self.gamma = gamma
        self.epsilon = epsilon
        self.all_data = env.all_data

    def step(self, action):
        """
        Takes a step, stores the reward, and returns a scaled reward.
        If the episode ends, it updates the running statistics.
        """
        obs, reward, terminated, truncated, info = self.env.step(action)

        self.episode_rewards.append(reward)

        if terminated or truncated:
            self._update_reward_stats()
            # Clear the buffer for the next episode
            self.episode_rewards.clear()

        # Scale the reward by the standard deviation of the returns
        scaled_reward = reward / np.sqrt(self.return_rms.var + self.epsilon)

        return obs, scaled_reward, terminated, truncated, info

    def reset(self, **kwargs):
        """Resets the environment and clears the reward buffer."""
        self.episode_rewards.clear()
        return self.env.reset(**kwargs)

    def _update_reward_stats(self):
        """
        Calculates discounted returns from the episode buffer and updates the
        running mean and standard deviation.
        """
        # Convert deque to a list for easier processing
        rewards = list(self.episode_rewards)

        # Calculate discounted returns (reward-to-go)
        discounted_returns = np.zeros_like(rewards, dtype=np.float64)
        running_add = 0
        for i in reversed(range(len(rewards))):
            running_add = rewards[i] + self.gamma * running_add
            discounted_returns[i] = running_add

        # Update the running statistics with the returns from this episode
        self.return_rms.update(discounted_returns)

In [3]:
%load_ext autoreload
%autoreload 2

### Global Variables
DATA_SAVE_PATH = "../../../models/used_data"
DAYS_PER_EPISODE = 3
TRAIN_TEST_SPLIT_FRACTION = 0.2  # Amount of data you reserve for testing.
BUFFER_DAYS = 3  # Amount of days that needs to be between the test episodes.

In [4]:
from Simulation.suite_simple_trading.pre_processing import clean_data
import pandas as pd
import os
from Simulation.suite_simple_trading.data_splitting import get_or_create_train_test_split

### Getting data

raw_data_path = '../../../data/2025_minute.csv'
cleaned_data_cache_path = '../../../data/2025_minute_cleaned.pkl'

if os.path.exists(cleaned_data_cache_path):
    print(f"Loading cached cleaned data from: {cleaned_data_cache_path}")
    cleaned_df = pd.read_pickle(cleaned_data_cache_path)
else:
    print("No cached data found. Running the full cleaning process...")
    raw_df = pd.read_csv(raw_data_path, sep=';')
    cleaned_df = clean_data(raw_df)
    print(f"Saving cleaned data to cache: {cleaned_data_cache_path}")
    cleaned_df.to_pickle(cleaned_data_cache_path)

all_data = cleaned_df[['Datetime', 'Imbalance Price']]
train_df, test_df, nr_of_episodes = get_or_create_train_test_split(
    all_data=all_data,
    save_path=DATA_SAVE_PATH,
    days_per_episode=DAYS_PER_EPISODE,
    test_fraction=TRAIN_TEST_SPLIT_FRACTION,
    buffer_days=BUFFER_DAYS
)

Loading cached cleaned data from: ../../../data/2025_minute_cleaned.pkl
--- Loading existing train/test data from '../../../models/used_data' ---
--- Data loaded successfully. Found 19 test episodes. ---
Test Data Size / Train Data Size: 0.25


In [7]:
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib import MaskablePPO
from Simulation.suite_simple_trading.observation_wrappers import RobustScalingWrapper
### The model (Environment) used in all agents

from Simulation.suite_simple_trading.model import ExtendedBatteryEnv, BaseBatteryEnv

train_env = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=train_df,
    days_per_episode=DAYS_PER_EPISODE
)
train_env_scaled_obs = RobustScalingWrapper(train_env)
train_env_fully_scaled = NormalizeRewardWrapper(train_env_scaled_obs, gamma=0.99)
ppo_agent = MaskablePPO(MaskableActorCriticPolicy, train_env_fully_scaled)
ppo_agent.learn(total_timesteps=200_000, progress_bar=True)


Output()

--- Initializing Robust Observation Scaling Wrapper (Method: robust) ---
Price (Robust): Median=78.17, IQR=87.78
--- Wrapper ready. ---


In [8]:
from Simulation.suite_simple_trading.policy import PPOAgentDecisionMaker
from Simulation.suite_simple_trading.simulation import run_evaluation

### Testing
test_env = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=test_df,
    days_per_episode=DAYS_PER_EPISODE
)
test_env_scaled = RobustScalingWrapper(test_env)
test_env_fully_scaled = NormalizeRewardWrapper(test_env_scaled, gamma=0.99)
decision_maker_PPO = PPOAgentDecisionMaker(ppo_agent, test_env_fully_scaled)

result_ppo = run_evaluation(test_env_fully_scaled,
                            ppo_agent, number_of_episodes=nr_of_episodes)
print(f"reward ppo: {sum(result_ppo["real_rewards"])}")

--- Initializing Robust Observation Scaling Wrapper (Method: robust) ---
Price (Robust): Median=78.92, IQR=91.47
--- Wrapper ready. ---
Starting episode 1/19
From 2025-01-03 00:00:00+00:00 to 2025-01-05 23:59:00+00:00
Finished with total (scaled) reward: 9674.72
Starting episode 2/19
From 2025-01-16 00:00:00+00:00 to 2025-01-18 23:59:00+00:00
Finished with total (scaled) reward: 5.51
Starting episode 3/19
From 2025-01-29 00:00:00+00:00 to 2025-01-31 23:59:00+00:00
Finished with total (scaled) reward: 32.19
Starting episode 4/19
From 2025-02-05 00:00:00+00:00 to 2025-02-07 23:59:00+00:00
Finished with total (scaled) reward: 10.84
Starting episode 5/19
From 2025-02-14 00:00:00+00:00 to 2025-02-16 23:59:00+00:00
Finished with total (scaled) reward: 6.13
Starting episode 6/19
From 2025-03-07 00:00:00+00:00 to 2025-03-09 23:59:00+00:00
Finished with total (scaled) reward: 41.87
Starting episode 7/19
From 2025-03-31 00:00:00+00:00 to 2025-04-02 23:59:00+00:00
Finished with total (scaled) rew